In [1]:
# -*- coding: utf-8 -*-

# Phase 5 — 제안 모델 (v4 신규)

Phase 1~4는 **분석**이다. 이 노트북이 **본선 당일 학습시킬 모델**의 원형이다.

서비스가 콜 등록 시점에 답해야 하는 것은 두 가지다.

| 모델 | 질문 | 학습 대상 |
|---|---|---|
| **A. 수락 예측** | 이 화주가 이 제안을 받아들일까 | 개입제안로그 3,117건(v5, 레버 6종) |
| **B. 반사실 결과 예측** | 조건을 바꾸면 운임·배차시간이 어떻게 되나 | 콜등록이력 |

---
## 설계 원칙 3가지

### ① 레버별 모델 4개 대신 `제안유형`을 피처로 넣은 단일 모델

분할 레버는 112건에 수락 9건뿐이다. 따로 떼어 분류기를 만들면 적합이 안 된다.
단일 모델로 가면 분할 112건이 전체 2,560건이 공유하는 모델에 "분할은 수락률이 낮은
레버"라는 신호로 기여한다. **6-2에서 두 방식을 실제로 비교해 이걸 실증한다.**

### ② 누출 차단 3종

- `화주프로파일` 시트(생성 파라미터 정답지)는 어떤 형태로도 피처에 넣지 않는다
- `차종`을 통째로 넣지 않고 `적재형태` + `톤급`으로 분해한다
- `화주ID`를 범주형으로 넣지 않는다 (품목 우회 경로)

추가로 **같은 콜의 제안이 훈련/검증에 흩어지면 안 된다.** 한 콜의 제안 3개는 화주
특성을 공유하므로 무작위 분할하면 검증 성능이 부풀려진다. → 콜 단위 그룹 분할.

### ③ 반사실 검증은 진짜 반사실로 한다

모델 B를 **개입이 적용되지 않은 콜로만 학습**시킨 뒤, 제안이 수락돼 조건이 실제로
바뀐 콜에 적용한다. 그 콜들은 바뀐 조건의 실제 결과(`실제_체결운임`)를 알고 있으므로
**"조건을 바꿨다면"의 예측을 실측과 대조할 수 있다.** 합성 데이터의 이점이다.

In [2]:
import os
import json
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from dotenv import load_dotenv
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold, KFold, cross_val_predict, RandomizedSearchCV
from sklearn.metrics import roc_auc_score, brier_score_loss, average_precision_score
from sklearn.inspection import permutation_importance
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

for _f in ["Noto Sans CJK KR", "Noto Sans CJK JP", "AppleGothic", "Malgun Gothic", "DejaVu Sans"]:
    if any(_f in f.name for f in matplotlib.font_manager.fontManager.ttflist):
        plt.rcParams["font.family"] = _f
        break
plt.rcParams["axes.unicode_minus"] = False

load_dotenv()
SRC = Path(os.getenv("DATA_DIR", ".")).expanduser().resolve()
OUT = SRC / "out"; OUT.mkdir(exist_ok=True)
FIG = OUT / "fig"; FIG.mkdir(exist_ok=True)
MODEL_DIR = OUT / "models"; MODEL_DIR.mkdir(exist_ok=True)   # 학습된 모델 저장 위치 — .env의 DATA_DIR 기준
SEED = 42

_pv = OUT / "_provenance.json"
if _pv.exists():
    print(f"[출처] {json.loads(_pv.read_text(encoding='utf-8'))['source']}")

calls = pd.read_csv(OUT / "phase1_분석테이블.csv", parse_dates=["등록일시"])
prop = pd.read_csv(OUT / "phase1_제안로그.csv")
seg = pd.read_csv(OUT / "phase3_화주세그먼트.csv", index_col=0)

# 화주 특성은 '정답지'가 아니라 과거 발주에서 집계한 값만 쓴다
calls = calls.merge(seg[["세그먼트_규칙", "발주건수", "요일집중도", "노선_집중도"]],
                    left_on="화주ID", right_index=True, how="left")
print(f"[load] 콜 {len(calls)} / 제안 {len(prop)}")

BAN = ["화주ID", "차종", "콜ID", "개입적용", "인상이력"]
print(f"[누출 차단] 피처에서 제외: {BAN}")

[출처] 가상데이터-최종.xlsx
[load] 콜 12000 / 제안 9324
[누출 차단] 피처에서 제외: ['화주ID', '차종', '콜ID', '개입적용', '인상이력']


## 6-1. 모델 A 데이터 — 제안 단위

콜 속성 + 제안 속성을 결합한다. `예측_수락가능`·`예측_배차분`·`예측_운임`은
**제안 시점에 시스템이 이미 계산해 화면에 띄우는 값**이므로 피처로 써도 누출이 아니다.
반면 `실제_체결운임`·`실제_배차분`은 수락 이후에만 생기므로 절대 넣으면 안 된다.

### 정확도 개선 — 화주별 과거 수락률 (대리 피처)

6-2b에서 보듯 모델 A는 오라클 상한(0.66대)에 못 미친다. 갭의 원인은 `유연성향`
(생성기에서 수락확률 자체를 만드는 잠재변수, `수락확률=레버기본률×(0.5+유연성향)`)이
관측 불가능하기 때문이다. 지금까지의 대리신호(`flex_cargo`·`flex_veh`)는 콜 하나의
속성일 뿐 "이 화주가 실제로 과거에 어떻게 반응했는가"를 보지 않는다. **화주별 과거
수락률**이 유연성향에 가장 직접 연결된 관측 가능 신호라 피처로 추가한다 —
leave-one-call-out(자기 콜의 제안은 제외) + 베이지안 스무딩(표본이 적은 화주는
전체평균 쪽으로 수축)으로 계산한다. `화주ID`를 범주형으로 넣는 것과 달리 식별자가
아니라 `발주건수`처럼 행동을 요약한 집계치라 누출 차단 원칙에 어긋나지 않는다.

In [3]:
CALL_FEATS = ["시간창_분", "리드타임_h", "거리km", "톤급", "중량_kg", "파렛트", "적재율",
              "긴급플래그", "권한_미승인", "flex_cargo", "flex_veh",
              "기준운임", "제시운임", "수락가능_현조건", "수락가능_시간완화",
              "시간변경비용_원_시간", "발주건수", "요일집중도", "노선_집중도"]
CAT_FEATS = ["적재형태", "세그먼트_규칙", "등록주체", "상차방법", "지불방식"]

A = prop.merge(calls[["콜ID", "화주ID"] + CALL_FEATS + CAT_FEATS], on="콜ID", how="left")
A["수락"] = (A["사용자반응"] == "수락").astype(int)

# 화주별 과거 수락률 — leave-one-call-out(자기 콜 제외) + 베이지안 스무딩.
# 화주ID 자체는 BAN 목록대로 피처에 안 넣는다 — 식별자가 아니라 행동 집계 신호만 남긴다.
# (발주건수·요일집중도·노선_집중도와 같은 패턴이다.) 유연성향은 수락확률 자체를 만드는
# 공식(PACC×(0.5+유연성향))에 들어가므로, 화주의 과거 수락 이력이 이론적으로 가장 직접적인
# 대리 신호다 — 6-2b 오라클 상한(0.663~0.674)에 가장 가깝게 붙을 수 있는 카드.
_call_n = A.groupby(["화주ID", "콜ID"])["수락"].transform("size")
_call_sum = A.groupby(["화주ID", "콜ID"])["수락"].transform("sum")
_shipper_n = A.groupby("화주ID")["수락"].transform("size")
_shipper_sum = A.groupby("화주ID")["수락"].transform("sum")
_GLOBAL_RATE, _PRIOR_W = A["수락"].mean(), 10   # PRIOR_W건치 가상표본만큼 전체평균 쪽으로 수축
A["화주_과거수락률"] = ((_shipper_sum - _call_sum) + _PRIOR_W * _GLOBAL_RATE) / \
                    ((_shipper_n - _call_n) + _PRIOR_W)

XA = A[CALL_FEATS + ["예측_수락가능", "예측_운임", "예측_배차분", "신뢰도", "제안순위",
                      "화주_과거수락률"]].copy()
XA["log_시간창"] = np.log1p(A["시간창_분"])
XA["후보배율"] = A["수락가능_시간완화"] / A["수락가능_현조건"].clip(lower=1)
XA["운임변화율"] = A["예측_운임"] / A["제시운임"] - 1        # 제안이 운임을 얼마나 바꾸나
XA["배차단축율"] = 1 - A["예측_배차분"] / (266 / A["수락가능_현조건"].clip(lower=1) ** 0.354)
for c in CAT_FEATS + ["제안유형"]:
    XA[c] = A[c].astype("category")
yA = A["수락"]
groups = A["콜ID"]

print(f"[모델 A] {XA.shape[0]}행 × {XA.shape[1]}피처 / 수락 {yA.sum()}건 ({yA.mean():.1%})")
print(f"  레버별: {dict(A.groupby('제안유형')['수락'].agg(['size','sum']).apply(tuple, axis=1))}")

[모델 A] 9324행 × 35피처 / 수락 1509건 (16.2%)
  레버별: {'가격': (539, 65), '권한': (439, 58), '날짜': (1254, 111), '분할': (381, 26), '시간': (3680, 911), '차종': (3031, 338)}


## 6-2. 단일 모델 vs 레버별 모델 — 설계 원칙 ①의 실증

콜 단위 GroupKFold 5겹. 같은 콜의 제안이 훈련/검증에 갈라지지 않게 묶는다.

`make_clf()`는 지금까지 손으로 고른 고정 하이퍼파라미터를 썼다(모델 B는 6-3에서
이미 `RandomizedSearchCV`로 튜닝했다). 여기서도 같은 방식 — `GroupKFold`를 그대로
탐색에 물려 콜 단위 분할을 유지한 채로 튜닝한다.

In [4]:
CLF_GRID_A = {
    "learning_rate": [0.02, 0.03, 0.05, 0.08, 0.1],
    "max_depth": [3, 4, 5, 6, None],
    "max_leaf_nodes": [15, 31, 63],
    "min_samples_leaf": [10, 20, 30, 50],
    "l2_regularization": [0.0, 0.1, 0.5, 1.0],
}
_search_a = RandomizedSearchCV(
    HistGradientBoostingClassifier(random_state=SEED, max_iter=1000, categorical_features="from_dtype",
                                   early_stopping=True, n_iter_no_change=15, validation_fraction=.15),
    CLF_GRID_A, n_iter=25, cv=GroupKFold(n_splits=5), scoring="roc_auc", random_state=SEED, n_jobs=1)
_search_a.fit(XA, yA, groups=groups)
BEST_A = _search_a.best_params_
print(f"[모델 A 하이퍼파라미터 탐색] {BEST_A}")


def make_clf():
    return HistGradientBoostingClassifier(
        random_state=SEED, max_iter=1000, categorical_features="from_dtype",
        early_stopping=True, n_iter_no_change=15, validation_fraction=.15, **BEST_A)


gkf = GroupKFold(n_splits=5)
pred_single = cross_val_predict(make_clf(), XA, yA, cv=gkf, groups=groups, method="predict_proba")[:, 1]

print("\n" + "=" * 64 + "\n[단일 모델] 제안유형을 피처로 — 교차검증\n" + "=" * 64)
print(f"  전체   AUC={roc_auc_score(yA, pred_single):.3f}  "
      f"PR-AUC={average_precision_score(yA, pred_single):.3f}  "
      f"Brier={brier_score_loss(yA, pred_single):.4f}")

rows = []
for lv in sorted(A["제안유형"].unique()):
    m = A["제안유형"] == lv
    n_pos = int(yA[m].sum())
    auc = roc_auc_score(yA[m], pred_single[m]) if 0 < n_pos < m.sum() else np.nan
    rows.append(dict(레버=lv, 표본=int(m.sum()), 수락=n_pos, 단일모델_AUC=round(auc, 3)))
    print(f"  {lv:4s} n={m.sum():5d} 수락={n_pos:4d}  AUC={auc:.3f}")

print("\n" + "=" * 64 + "\n[레버별 모델] 각 레버만으로 따로 학습\n" + "=" * 64)
for r in rows:
    lv = r["레버"]
    m = (A["제안유형"] == lv).values
    Xl, yl, gl = XA[m].drop(columns=["제안유형"]), yA[m], groups[m]
    if yl.sum() < 10 or yl.nunique() < 2:
        print(f"  {lv:4s} 수락 {int(yl.sum())}건 → **학습 불가** (양성 표본 부족)")
        r["레버별_AUC"] = np.nan
        r["비고"] = "학습 불가"
        continue
    k = min(5, int(yl.sum()), gl.nunique())
    p = cross_val_predict(make_clf(), Xl, yl, cv=GroupKFold(n_splits=k),
                          groups=gl, method="predict_proba")[:, 1]
    auc = roc_auc_score(yl, p)
    r["레버별_AUC"] = round(auc, 3)
    r["비고"] = ""
    print(f"  {lv:4s} n={m.sum():5d} 수락={int(yl.sum()):4d}  AUC={auc:.3f}  "
          f"(단일 {r['단일모델_AUC']:.3f} 대비 {auc - r['단일모델_AUC']:+.3f})")

cmp = pd.DataFrame(rows)
cmp.to_csv(OUT / "phase5_모델비교.csv", index=False, encoding="utf-8-sig")
print("\n[결론] 분할 레버는 단독 학습이 아예 불가능하다(양성 9건). 단일 모델에서는 최소한 적합된다.")
print("       다만 레버 내부 AUC는 0.5 근처다 — 왜 그런지는 바로 아래에서 상한을 재본다.")

[모델 A 하이퍼파라미터 탐색] {'min_samples_leaf': 30, 'max_leaf_nodes': 63, 'max_depth': None, 'learning_rate': 0.08, 'l2_regularization': 0.5}



[단일 모델] 제안유형을 피처로 — 교차검증
  전체   AUC=0.720  PR-AUC=0.375  Brier=0.1206
  가격   n=  539 수락=  65  AUC=0.539
  권한   n=  439 수락=  58  AUC=0.639
  날짜   n= 1254 수락= 111  AUC=0.605
  분할   n=  381 수락=  26  AUC=0.504
  시간   n= 3680 수락= 911  AUC=0.741
  차종   n= 3031 수락= 338  AUC=0.615

[레버별 모델] 각 레버만으로 따로 학습


  가격   n=  539 수락=  65  AUC=0.428  (단일 0.539 대비 -0.111)


  권한   n=  439 수락=  58  AUC=0.518  (단일 0.639 대비 -0.121)


  날짜   n= 1254 수락= 111  AUC=0.528  (단일 0.605 대비 -0.077)
  분할   n=  381 수락=  26  AUC=0.491  (단일 0.504 대비 -0.013)


  시간   n= 3680 수락= 911  AUC=0.705  (단일 0.741 대비 -0.036)


  차종   n= 3031 수락= 338  AUC=0.518  (단일 0.615 대비 -0.097)

[결론] 분할 레버는 단독 학습이 아예 불가능하다(양성 9건). 단일 모델에서는 최소한 적합된다.
       다만 레버 내부 AUC는 0.5 근처다 — 왜 그런지는 바로 아래에서 상한을 재본다.


### 6-2b. 예측 가능성의 상한 — 이 모델이 나쁜 건가, 원래 안 되는 건가

레버별 AUC가 0.5 근처면 보통 모델을 의심한다. 그런데 **생성 규칙을 알고 있으므로
상한을 직접 계산할 수 있다.**

생성기에서 수락은 `u < 수락확률`, `수락확률 = 레버기본률 × (0.5 + 유연성향)`이다.
`유연성향`은 화주프로파일(정답지)에만 있고 모델 입력이 금지된 잠재변수다.

그래서 **유연성향을 그대로 넣은 '오라클'의 AUC가 곧 이론적 상한**이다.
이걸 넘는 모델은 존재할 수 없다. 오라클조차 낮다면 목표 자체가 원래 어려운 것이다.

> 이 절은 **진단이지 모델이 아니다.** 정답지를 쓰므로 어떤 성능 수치로도 인용하면 안 된다.

In [5]:
try:
    _truth = pd.read_excel(SRC / os.getenv("DATA_FILE", "유연오더_가상데이터_v13.xlsx"),
                           sheet_name="화주프로파일")[["화주ID", "유연성향"]]
    _o = A.merge(_truth, on="화주ID")     # 화주ID는 이미 A에 있다 (6-1에서 대리 피처 계산용으로 편입)
    PA = {"시간": .28, "차종": .18, "가격": .15, "분할": .08, "권한": .15, "날짜": .12}     # 생성기 레버 기본 수락률
    _o["오라클확률"] = _o["제안유형"].map(PA) * (0.5 + _o["유연성향"])
    print(f"\n  오라클 전체 AUC = {roc_auc_score(_o['수락'], _o['오라클확률']):.3f}"
          f"   (우리 단일 모델 {roc_auc_score(yA, pred_single):.3f})")
    for lv in sorted(_o["제안유형"].unique()):
        m = _o["제안유형"] == lv
        if _o.loc[m, "수락"].nunique() > 1:
            print(f"    {lv:4s} 오라클 AUC = {roc_auc_score(_o.loc[m,'수락'], _o.loc[m,'오라클확률']):.3f}")
        else:
            print(f"    {lv:4s} 오라클 AUC = 계산 불가 (수락 {int(_o.loc[m,'수락'].sum())}건 — 완전 동질)")
    print("\n  [관측 가능한 대리변수와 유연성향의 상관]")
    for col in ["flex_cargo", "flex_veh", "시간창_분", "화주_과거수락률"]:
        print(f"    {col:14s} r = {np.corrcoef(_o[col], _o['유연성향'])[0,1]:+.3f}")
    print("\n  → 수락은 잠재 성향에 따른 확률 추첨이고, 그 성향을 관측 데이터로는 절반밖에 못 본다.")
    print("     레버 내부 AUC가 0.5 근처인 것은 모델 결함이 아니라 상한이 거기에 있기 때문이다.")
    print("     ★ 서비스 가치는 '누가 수락할지 맞히기'(모델 A)보다 ")
    print("       '수락하면 결과가 어떻게 되는지'(모델 B)에 있다. 아래 6-3~6-4가 본체다.")
except Exception as e:
    print(f"\n  [상한 진단] 화주프로파일 없음 — 생략 ({e})")


  오라클 전체 AUC = 0.658   (우리 단일 모델 0.720)
    가격   오라클 AUC = 0.527
    권한   오라클 AUC = 0.548
    날짜   오라클 AUC = 0.561
    분할   오라클 AUC = 0.541
    시간   오라클 AUC = 0.587
    차종   오라클 AUC = 0.559

  [관측 가능한 대리변수와 유연성향의 상관]
    flex_cargo     r = +0.453
    flex_veh       r = +0.265
    시간창_분          r = -0.066
    화주_과거수락률       r = +0.464

  → 수락은 잠재 성향에 따른 확률 추첨이고, 그 성향을 관측 데이터로는 절반밖에 못 본다.
     레버 내부 AUC가 0.5 근처인 것은 모델 결함이 아니라 상한이 거기에 있기 때문이다.
     ★ 서비스 가치는 '누가 수락할지 맞히기'(모델 A)보다 
       '수락하면 결과가 어떻게 되는지'(모델 B)에 있다. 아래 6-3~6-4가 본체다.


In [6]:
# 최종 모델 적합 + 중요도
clf = make_clf().fit(XA, yA)
tr, te = next(GroupKFold(n_splits=5).split(XA, yA, groups))
clf_h = make_clf().fit(XA.iloc[tr], yA.iloc[tr])
pi = permutation_importance(clf_h, XA.iloc[te], yA.iloc[te], n_repeats=10,
                            random_state=SEED, scoring="roc_auc")
imp = pd.Series(pi.importances_mean, index=XA.columns).sort_values(ascending=False)
print("\n[모델 A 순열중요도 — 홀드아웃 상위 10]")
print(imp.head(10).round(4).to_string())
imp.round(5).to_frame("중요도").to_csv(OUT / "phase5_수락모델_중요도.csv", encoding="utf-8-sig")


[모델 A 순열중요도 — 홀드아웃 상위 10]
화주_과거수락률       0.1261
배차단축율          0.0288
요일집중도          0.0192
노선_집중도         0.0176
제안유형           0.0161
발주건수           0.0149
세그먼트_규칙        0.0089
시간변경비용_원_시간    0.0062
적재형태           0.0036
거리km           0.0028


In [7]:
# 모델 A 저장 — 전체 데이터로 적합된 clf(제안 수락 예측)를 재사용 가능하게 남긴다.
# 추론 시 필요한 피처 목록·범주형 컬럼·튜닝 하이퍼파라미터를 함께 묶는다.
MODEL_A_PATH = MODEL_DIR / "model_A_수락예측.joblib"
joblib.dump({
    "model": clf,
    "features": list(XA.columns),
    "cat_feats": CAT_FEATS + ["제안유형"],
    "best_params": BEST_A,
}, MODEL_A_PATH)
print(f"[저장] 모델 A → {MODEL_A_PATH}")

[저장] 모델 A → /Volumes/SSD/공모전자료/유통_물류-해커톤/데이터셋-모음/out/models/model_A_수락예측.joblib


## 6-3. 모델 B — 반사실 결과 예측

**개입이 적용되지 않은 3,592건으로만 학습한다.** 개입 수락 콜은 조건이 바뀐 뒤의
결과가 들어 있어 학습에 쓰면 검증이 오염된다.

두 개를 예측한다.

- `log(체결배율)` — 성사 건 대상 회귀
- `유찰` — 조건이 나쁘면 애초에 안 잡힌다. 기대운임을 내려면 둘을 곱해야 한다

### 정확도 개선 4가지

- **미사용 피처 편입**: `phase1_분석테이블`에 이미 계산돼 있었지만 안 쓰던
  `노선ID`·`요일`·`시간대`·`주말상차`를 추가한다. 전부 콜 등록 시점에 이미 확정된
  값이라 누출이 아니다.
- **선택편향 보정**: 운임은 성사된 건만 관측되는 반사실이다("유찰 안 났다면 얼마에
  잡혔을까"가 아니라 "유찰 안 난 건들의 운임"만 학습). 유찰모델의 교차검증
  **out-of-fold** 확률을 "이 조건이면 애초에 안 잡힐 위험"이라는 피처로 운임
  회귀에 넣어 두 모델을 연결한다 — 자기 자신을 학습에 쓴 예측이 아니므로 누출이
  아니다.
- **하이퍼파라미터 탐색**: 지금까지 `HistGradientBoosting*`은 손으로 고른 고정값을
  썼다. `RandomizedSearchCV` + `early_stopping`으로 두 모델 모두 재탐색한다.
- **불확실성 구간 + 순열중요도(6-3b)**: 경매가 확률적이라 점추정 하나로는 부족하다.
  같은 튜닝 파라미터로 손실만 분위수로 바꿔 P10~P90 구간을 추가로 낸다. 어떤
  피처가 실제로 기여하는지도 홀드아웃 순열중요도로 확인한다(모델 A에는 있었지만
  모델 B에는 없던 진단이다).

In [8]:
pure = calls[calls["개입수락"] == 0].copy()
COND = ["시간창_분", "리드타임_h", "거리km", "톤급", "중량_kg", "파렛트", "적재율",
        "긴급플래그", "권한_미승인", "flex_cargo", "flex_veh", "기준운임",
        "수락가능_현조건", "수락가능_시간완화", "발주건수", "요일집중도", "노선_집중도",
        "시간대", "주말상차"]
CATB = ["적재형태", "세그먼트_규칙", "등록주체", "노선ID", "요일"]


def designB_base(d):
    X = d[COND].copy()
    X["log_시간창"] = np.log1p(d["시간창_분"])
    X["log_리드타임"] = np.log(d["리드타임_h"].clip(lower=1))
    for c in CATB:
        X[c] = d[c].astype("category")
    return X


REG_GRID = {
    "learning_rate": [0.02, 0.03, 0.05, 0.08, 0.1],
    "max_depth": [3, 4, 5, 6, None],
    "max_leaf_nodes": [15, 31, 63],
    "min_samples_leaf": [10, 20, 30, 50],
    "l2_regularization": [0.0, 0.1, 0.5, 1.0],
    "loss": ["squared_error", "absolute_error"],
}
CLF_GRID = {k: v for k, v in REG_GRID.items() if k != "loss"}


def tune(base_estimator, grid, X, y, scoring):
    search = RandomizedSearchCV(base_estimator, grid, n_iter=25, cv=5, scoring=scoring,
                                random_state=SEED, n_jobs=1)
    search.fit(X, y)
    return search.best_estimator_, search.best_params_


# --- 유찰모델을 먼저 학습 — 아래 운임 회귀의 선택편향 보정 피처로 OOF 확률을 재사용 ---
XF, yF = designB_base(pure), pure["유찰"]
clfF, best_f = tune(
    HistGradientBoostingClassifier(random_state=SEED, max_iter=1000, categorical_features="from_dtype",
                                   early_stopping=True, n_iter_no_change=15, validation_fraction=.15),
    CLF_GRID, XF, yF, "roc_auc")
cvF = cross_val_predict(clfF, XF, yF, cv=5, method="predict_proba")[:, 1]
print(f"[모델 B-유찰] n={len(yF)}  CV AUC={roc_auc_score(yF, cvF):.3f}  "
      f"Brier={brier_score_loss(yF, cvF):.4f}")
print(f"  튜닝 파라미터: {best_f}")
clfF.fit(XF, yF)

# --- 운임 회귀: 유찰모델의 OOF 확률을 "이 조건이면 애초에 안 잡힐 위험"으로 피처에 추가 ---
ok_mask = (pure["결과"] == "성사").values
okp = pure[ok_mask]
XB = designB_base(okp).copy()
XB["유찰확률"] = cvF[ok_mask]
yB = np.log(okp["체결배율"])

regB, best_b = tune(
    HistGradientBoostingRegressor(random_state=SEED, max_iter=1000, categorical_features="from_dtype",
                                  early_stopping=True, n_iter_no_change=15, validation_fraction=.15),
    REG_GRID, XB, yB, "r2")
cvB = cross_val_predict(regB, XB, yB, cv=5)
print(f"\n[모델 B-운임] n={len(yB)}  CV R²={1 - ((yB - cvB)**2).sum() / ((yB - yB.mean())**2).sum():.3f}  "
      f"MAE(배율)={np.abs(np.exp(cvB) - np.exp(yB)).mean():.4f}")
print(f"  튜닝 파라미터: {best_b}")
regB.fit(XB, yB)


def designB(d):
    """추론 시점: 확정 학습된 clfF로 유찰확률을 예측해 선택편향 보정 피처로 포함한다"""
    Xb = designB_base(d)
    fail_prob = clfF.predict_proba(Xb)[:, 1]
    Xb = Xb.copy()
    Xb["유찰확률"] = fail_prob
    return Xb


[모델 B-유찰] n=10548  CV AUC=0.747  Brier=0.0838
  튜닝 파라미터: {'min_samples_leaf': 20, 'max_leaf_nodes': 63, 'max_depth': 3, 'learning_rate': 0.02, 'l2_regularization': 0.5}



[모델 B-운임] n=9495  CV R²=0.345  MAE(배율)=0.1037
  튜닝 파라미터: {'min_samples_leaf': 20, 'max_leaf_nodes': 31, 'max_depth': 3, 'loss': 'squared_error', 'learning_rate': 0.08, 'l2_regularization': 0.5}


### 6-3b. 불확실성 구간 + 순열중요도

지금까지 운임은 점추정 하나만 냈다. 경매 메커니즘 자체가 확률적이라 "정확히 얼마"는
원래 불가능하고, "대략 이 범위"가 더 정직한 답이다. 같은 튜닝 파라미터를 재사용해
손실만 분위수로 바꿔 구간을 추가로 적합한다.

**분위수 명목값을 그대로 믿지 않는다.** P10~P90이라고 학습시킨 구간이 실제로 80%를
담는다는 보장은 없다 — HGB의 분위수 손실은 근사이지 정확한 분위수 추정이 아니다.
그래서 P10/P90부터 P20/P80까지 후보 몇 개를 CV로 직접 재보고, **목표 80%
커버리지를 만족하는 것 중 가장 좁은 쌍**을 고른다(과소보다 과대커버가 안전하므로
80% 밑으로는 내려가지 않는다).

순열중요도는 모델 A(6-2 이후)에는 있었지만 모델 B에는 아직 없었다 — 홀드아웃에서
같은 방식으로 확인한다.

In [9]:
Q_PARAMS = {k: v for k, v in best_b.items() if k != "loss"}
TARGET_COVER = .80


def make_q(q):
    return HistGradientBoostingRegressor(random_state=SEED, max_iter=1000, categorical_features="from_dtype",
                                         early_stopping=True, n_iter_no_change=15, validation_fraction=.15,
                                         loss="quantile", quantile=q, **Q_PARAMS)


print("[모델 B-운임 구간 보정] 분위수 명목값 vs 실측 CV 커버리지")
_calib = []
for qlo, qhi in [(.10, .90), (.125, .875), (.15, .85), (.175, .825), (.20, .80)]:
    _lo, _hi = cross_val_predict(make_q(qlo), XB, yB, cv=5), cross_val_predict(make_q(qhi), XB, yB, cv=5)
    _cover = ((yB >= _lo) & (yB <= _hi)).mean()
    _width = np.exp(np.median(_hi - _lo))
    _calib.append((qlo, qhi, _cover, _width))
    print(f"  P{qlo*100:>4.1f}~P{qhi*100:<4.1f}  커버리지={_cover:.1%}  구간폭(배율)={_width:.3f}")

_ok = [c for c in _calib if c[2] >= TARGET_COVER]
Q_LO, Q_HI, _, _ = min(_ok, key=lambda c: c[3]) if _ok else max(_calib, key=lambda c: c[2])
print(f"[선택] P{Q_LO*100:.1f}~P{Q_HI*100:.1f} — 목표({TARGET_COVER:.0%}) 충족 중 가장 좁은 구간"
      if _ok else f"[선택] P{Q_LO*100:.1f}~P{Q_HI*100:.1f} — 어떤 후보도 목표 미달, 최대 커버리지로 대체")

regB_lo, regB_hi = make_q(Q_LO), make_q(Q_HI)
regB_lo.fit(XB, yB)
regB_hi.fit(XB, yB)

# --- 순열중요도 — 홀드아웃에서 (훈련셋 중요도는 과적합을 반영) ---
kf5 = KFold(n_splits=5, shuffle=True, random_state=SEED)
trB, teB = next(kf5.split(XB))
regB_h = clone(regB).fit(XB.iloc[trB], yB.iloc[trB])
impB = pd.Series(permutation_importance(regB_h, XB.iloc[teB], yB.iloc[teB], n_repeats=10,
                                        random_state=SEED, scoring="r2").importances_mean,
                 index=XB.columns).sort_values(ascending=False)
print("\n[모델 B-운임 순열중요도 — 홀드아웃 상위 10]")
print(impB.head(10).round(4).to_string())

trF, teF = next(kf5.split(XF))
clfF_h = clone(clfF).fit(XF.iloc[trF], yF.iloc[trF])
impF = pd.Series(permutation_importance(clfF_h, XF.iloc[teF], yF.iloc[teF], n_repeats=10,
                                        random_state=SEED, scoring="roc_auc").importances_mean,
                 index=XF.columns).sort_values(ascending=False)
print("\n[모델 B-유찰 순열중요도 — 홀드아웃 상위 10]")
print(impF.head(10).round(4).to_string())

[모델 B-운임 구간 보정] 분위수 명목값 vs 실측 CV 커버리지


  P10.0~P90.0  커버리지=89.3%  구간폭(배율)=1.289


  P12.5~P87.5  커버리지=87.4%  구간폭(배율)=1.272


  P15.0~P85.0  커버리지=85.7%  구간폭(배율)=1.269


  P17.5~P82.5  커버리지=82.7%  구간폭(배율)=1.265


  P20.0~P80.0  커버리지=79.6%  구간폭(배율)=1.245
[선택] P17.5~P82.5 — 목표(80%) 충족 중 가장 좁은 구간



[모델 B-운임 순열중요도 — 홀드아웃 상위 10]
긴급플래그        0.2674
리드타임_h       0.0605
유찰확률         0.0484
수락가능_현조건     0.0199
수락가능_시간완화    0.0048
노선ID         0.0041
요일집중도        0.0016
노선_집중도       0.0013
flex_veh     0.0005
주말상차         0.0000



[모델 B-유찰 순열중요도 — 홀드아웃 상위 10]
리드타임_h      0.2146
수락가능_현조건    0.0406
긴급플래그       0.0267
시간대         0.0048
기준운임        0.0019
적재율         0.0014
요일          0.0010
파렛트         0.0009
권한_미승인      0.0009
노선_집중도      0.0008


In [10]:
# 모델 B 저장 — 유찰분류기(clfF) + 운임회귀 3종(점추정 regB, 하한 regB_lo, 상한 regB_hi).
# designB()가 clfF를 클로저로 참조하므로, 추론 쪽에서 그대로 쓰려면 네 모델을 항상 같이 로드해야 한다.
MODEL_B_PATH = MODEL_DIR / "model_B_반사실예측.joblib"
joblib.dump({
    "유찰모델": clfF,
    "운임모델": regB,
    "운임모델_하한": regB_lo,
    "운임모델_상한": regB_hi,
    "features_유찰": list(XF.columns),
    "features_운임": list(XB.columns),
    "cat_feats": CATB,
    "quantile_lo": Q_LO,
    "quantile_hi": Q_HI,
    "best_params_유찰": best_f,
    "best_params_운임": best_b,
}, MODEL_B_PATH)
print(f"[저장] 모델 B → {MODEL_B_PATH}")

[저장] 모델 B → /Volumes/SSD/공모전자료/유통_물류-해커톤/데이터셋-모음/out/models/model_B_반사실예측.joblib


## 6-4. 반사실 검증 — "조건을 바꿨다면" 예측을 실측과 대조

제안이 수락돼 조건이 실제로 바뀐 콜을 가져와, **바뀐 조건을 모델에 넣고** 결과를 예측한 뒤
실제 값과 비교한다. 학습에 한 번도 쓰지 않은 콜들이다.

조건 변경 규칙은 생성기와 동일하게 맞춘다.

| 레버 | 바뀌는 것 |
|---|---|
| 시간 | `시간창_분` → 2880, `수락가능_현조건` → `수락가능_시간완화` |
| 차종 | `flex_veh` → 1 (풀 필터가 호환으로 열린다) |
| 권한(v5) | `시간창_분` → `개입후_시간창_분`(승인으로 복원된 실측값), `권한_미승인` → 0. `수락가능_현조건`은 생성기의 `navail`이 가용위상(균등분포) 임계값 방식이라 창 폭에 선형이므로, 두 관측점(현조건·시간완화) 사이를 선형보간하면 정확한 값이 나온다 |
| 날짜(v5) | `리드타임_h` → `개입후_리드타임_h`, `긴급플래그` → 0 (절대 임계 15h를 하루 연기로 반드시 넘긴다) |
| 가격 · 분할 | 기준운임 자체가 바뀌어 배율 정의가 달라진다 → 이번 검증에서 제외 |

In [11]:
acc = prop[prop["사용자반응"] == "수락"].merge(
    calls, on="콜ID", how="left", suffixes=("_제안", ""))
acc = acc[acc["실제_체결운임"].notna()]
print(f"[검증 대상] 수락된 제안 {len(acc)}건 중 레버별: {dict(acc['제안유형'].value_counts())}")

target = acc[acc["제안유형"].isin(["시간", "차종", "권한", "날짜"])].copy()
cf = target.copy()
_t = cf["제안유형"] == "시간"
_v = cf["제안유형"] == "차종"
_a = cf["제안유형"] == "권한"
_d = cf["제안유형"] == "날짜"

# 등록 시점 조건(시간창_분)은 v4에서 덮어쓰지 않으므로 여기서 직접 반사실 값을 넣는다
cf.loc[_t, "시간창_분"] = 2880
cf.loc[_t, "flex_time"] = 1.0
cf.loc[_t, "수락가능_현조건"] = cf.loc[_t, "수락가능_시간완화"]
cf.loc[_v, "flex_veh"] = 1

# 권한(v5) — 승인으로 원래 시간창(개입후_시간창_분)이 복원된다.
# navail은 가용위상(균등분포)에 대한 선형 임계값(w_h/48)이라, 두 관측점
# (수락가능_현조건@0.5h · 수락가능_시간완화@48h) 사이를 선형보간하면 정확한 값이 나온다.
# 후보수는 정수 카운트라 보간 결과를 반올림해 int로 되돌린다.
cf.loc[_a, "권한_미승인"] = 0
_w0, _w1 = 0.5, cf.loc[_a, "개입후_시간창_분"] / 60
cf.loc[_a, "수락가능_현조건"] = (cf.loc[_a, "수락가능_현조건"]
    + (cf.loc[_a, "수락가능_시간완화"] - cf.loc[_a, "수락가능_현조건"]) * (_w1 - _w0) / (48 - _w0)
    ).round().astype(int)
cf.loc[_a, "시간창_분"] = cf.loc[_a, "개입후_시간창_분"]

# 날짜(v5) — 후보 수·창 폭은 그대로, 리드타임만 늘고 긴급 판정이 풀린다
cf.loc[_d, "리드타임_h"] = cf.loc[_d, "개입후_리드타임_h"]
cf.loc[_d, "긴급플래그"] = 0

Xcf = designB(cf)
pred_ratio = np.exp(regB.predict(Xcf))
pred_lo = np.exp(regB_lo.predict(Xcf))
pred_hi = np.exp(regB_hi.predict(Xcf))
actual_ratio = target["실제_체결운임"] / target["기준운임"]
base_ratio = target["제시운임"] / target["기준운임"]      # 아무것도 안 바꿨을 때의 출발점

res = pd.DataFrame({
    "콜ID": target["콜ID"].values, "레버": target["제안유형"].values,
    "실제_체결배율": actual_ratio.values, "예측_체결배율": pred_ratio,
    "예측_P10": pred_lo, "예측_P90": pred_hi,
    "오차": pred_ratio - actual_ratio.values,
})
res["절대오차%"] = (res["오차"].abs() / res["실제_체결배율"] * 100)
res["구간적중"] = (res["실제_체결배율"] >= res["예측_P10"]) & (res["실제_체결배율"] <= res["예측_P90"])

print(f"\n[반사실 예측 정확도] n={len(res)}")
print(f"  MAE(배율)      = {res['오차'].abs().mean():.4f}")
print(f"  MAPE           = {res['절대오차%'].mean():.2f}%")
print(f"  편향(평균 오차) = {res['오차'].mean():+.4f}  (0에 가까울수록 체계적 과대/과소 없음)")
print(f"  상관 r         = {np.corrcoef(res['예측_체결배율'], res['실제_체결배율'])[0,1]:.3f}")
print(f"  P10~P90 구간 커버리지(진짜 반사실, 목표 80%) = {res['구간적중'].mean():.1%}")
print(res.groupby("레버")[["오차", "절대오차%", "구간적중"]].agg({"오차": "mean", "절대오차%": "mean", "구간적중": "mean"}).round(4).to_string())

# 기준선 대비 — 아무 모델 없이 "조건을 바꿔도 그대로"라고 찍었을 때
naive = np.abs(base_ratio.values - actual_ratio.values).mean()
print(f"\n  기준선(변화 없음 가정) MAE = {naive:.4f}  →  모델이 "
      f"{(1 - res['오차'].abs().mean() / naive) * 100:.0f}% 개선")
res.round(4).to_csv(OUT / "phase5_반사실검증.csv", index=False, encoding="utf-8-sig")

[검증 대상] 수락된 제안 1452건 중 레버별: {'시간': np.int64(901), '차종': np.int64(304), '날짜': np.int64(108), '가격': np.int64(62), '권한': np.int64(54), '분할': np.int64(23)}

[반사실 예측 정확도] n=1367
  MAE(배율)      = 0.0744
  MAPE           = 6.44%
  편향(평균 오차) = -0.0046  (0에 가까울수록 체계적 과대/과소 없음)
  상관 r         = 0.606
  P10~P90 구간 커버리지(진짜 반사실, 목표 80%) = 85.8%
        오차   절대오차%    구간적중
레버                        
권한 -0.0180  9.4954  0.8148
날짜 -0.0117  7.5125  0.8611
시간 -0.0030  5.5400  0.8546
차종 -0.0043  8.1912  0.8750

  기준선(변화 없음 가정) MAE = 0.0935  →  모델이 20% 개선


## 6-5. 제안 카드 생성 — 서비스가 실제로 하는 일

콜 하나를 골라 적용 가능한 레버를 전부 돌려보고, **수락 확률 × 예상 결과**를 함께 낸다.
서비스 화면의 제안 카드가 이 출력이다. 미승인 콜은 생성기와 동일하게 권한 레버만
제안한다(다른 레버는 애초에 걸 자리가 없다).

In [12]:
def make_cards(call_row):
    """콜 1건에 대해 레버별 반사실 결과 + 수락 확률.
    미승인 콜은 생성기와 동일하게 권한 레버만 대상이다."""
    cards = []
    미승인 = call_row["권한_미승인"] == 1
    levers = ["권한"] if 미승인 else ["시간", "차종", "날짜"]
    for lv in levers:
        c = call_row.copy()
        if lv == "시간":
            c["시간창_분"] = 2880
            c["수락가능_현조건"] = c["수락가능_시간완화"]
            desc = f"창 {int(call_row['시간창_분'])}분 → 48시간"
        elif lv == "차종":
            c["flex_veh"] = 1
            desc = "단일 차종 → 호환 허용"
        elif lv == "날짜":
            c["리드타임_h"] = call_row["리드타임_h"] + 24
            c["긴급플래그"] = 0
            desc = f"상차일 1일 연기 (리드타임 {call_row['리드타임_h']:.0f}h→{call_row['리드타임_h']+24:.0f}h)"
        else:  # 권한 — 진짜 원래 선호 시간창(_원시간창)은 정답지라 파이프라인에 노출되지
               # 않는다. 과거 권한 승인 건들의 실측 복원값(중앙값)으로 대체한다.
            w1 = AUTH_RESTORE_MED / 60
            c["시간창_분"] = AUTH_RESTORE_MED
            c["권한_미승인"] = 0
            c["수락가능_현조건"] = int(round(call_row["수락가능_현조건"]
                + (call_row["수락가능_시간완화"] - call_row["수락가능_현조건"]) * (w1 - .5) / (48 - .5)))
            desc = f"원화주 승인 요청 → 시간창 0분→약 {int(AUTH_RESTORE_MED)}분 복원(추정)"
        d = pd.DataFrame([c])
        Xd = designB(d)
        ratio = float(np.exp(regB.predict(Xd))[0])
        ratio_lo = float(np.exp(regB_lo.predict(Xd))[0])
        ratio_hi = float(np.exp(regB_hi.predict(Xd))[0])
        pfail = float(clfF.predict_proba(designB_base(d))[0, 1])
        cards.append(dict(
            레버=lv, 내용=desc,
            예상체결운임=int(ratio * call_row["기준운임"]),
            예상범위=f"{int(ratio_lo * call_row['기준운임']):,}~{int(ratio_hi * call_row['기준운임']):,}",
            현재대비=f"{(ratio * call_row['기준운임'] / call_row['제시운임'] - 1) * 100:+.1f}%",
            유찰확률=f"{pfail:.1%}"))
    return pd.DataFrame(cards)


AUTH_RESTORE_MED = target.loc[target["제안유형"] == "권한", "개입후_시간창_분"].median()

sample = pure[(pure["시간창_분"] < 240) & (pure["결과"] == "성사") & (pure["권한_미승인"] == 0)].iloc[0]
print(f"\n[예시 콜] {sample['콜ID']} | {sample['노선ID']} {sample['품목']} "
      f"{sample['톤급']}t {sample['적재형태']} | 시간창 {int(sample['시간창_분'])}분 | "
      f"제시운임 {int(sample['제시운임']):,}원 → 실제체결 {int(sample['최종체결운임']):,}원")
print(make_cards(sample).to_string(index=False))

sample_auth = pure[pure["권한_미승인"] == 1].iloc[0]
print(f"\n[예시 콜·미승인] {sample_auth['콜ID']} | {sample_auth['노선ID']} {sample_auth['품목']} "
      f"{sample_auth['톤급']}t {sample_auth['적재형태']} | 시간창 0분(권한 게이트로 잠김)")
print(make_cards(sample_auth).to_string(index=False))


[예시 콜] C0000 | R10 섬유원단 5t 윙바디 | 시간창 0분 | 제시운임 384,000원 → 실제체결 384,000원
레버                       내용  예상체결운임            예상범위  현재대비 유찰확률
시간              창 0분 → 48시간  389633 384,000~400,941 +1.5% 0.5%
차종            단일 차종 → 호환 허용  420384 384,000~459,982 +9.5% 2.5%
날짜 상차일 1일 연기 (리드타임 47h→71h)  421099 384,000~459,982 +9.7% 2.0%

[예시 콜·미승인] C0006 | R05 섬유원단 5t 윙바디 | 시간창 0분(권한 게이트로 잠김)
레버                              내용  예상체결운임            예상범위  현재대비 유찰확률
권한 원화주 승인 요청 → 시간창 0분→약 60분 복원(추정)  357948 330,000~387,637 +8.5% 1.9%


## 시각화

In [13]:
fig, ax = plt.subplots(1, 3, figsize=(18, 5))

_c = cmp.set_index("레버")[["단일모델_AUC", "레버별_AUC"]]
_c.plot(kind="bar", ax=ax[0], color=["#3B6EA5", "#C25E5E"])
ax[0].axhline(.5, ls="--", c="gray", lw=1)
ax[0].set_title("6-2 단일 모델 vs 레버별 모델 (AUC)", fontsize=12)
ax[0].set_ylim(0, 1); ax[0].set_xlabel("")

imp.head(10)[::-1].plot(kind="barh", ax=ax[1], color="#5B8C5A")
ax[1].set_title("6-2 수락 예측 순열중요도 상위 10", fontsize=12)

ax[2].scatter(res["실제_체결배율"], res["예측_체결배율"], s=14, alpha=.5,
              c=np.where(res["레버"] == "시간", "#3B6EA5", "#C25E5E"))
_lo, _hi = res[["실제_체결배율", "예측_체결배율"]].values.min(), res[["실제_체결배율", "예측_체결배율"]].values.max()
ax[2].plot([_lo, _hi], [_lo, _hi], "k--", lw=1)
ax[2].set_xlabel("실제 체결배율"); ax[2].set_ylabel("예측 체결배율")
ax[2].set_title(f"6-4 반사실 검증 (MAPE {res['절대오차%'].mean():.1f}%)", fontsize=12)

plt.tight_layout()
plt.savefig(FIG / "phase5_요약.png", dpi=130, bbox_inches="tight")
print(f"\n[저장] {OUT}")


[저장] /Volumes/SSD/공모전자료/유통_물류-해커톤/데이터셋-모음/out


In [ ]:
# 서비스용 예측 저장 — 모델 B 직접 추론, 임의 톤급 가산/수동 보정 없음
TON_AXES, WINDOW_AXES = [5, 11, 25], [40, 120, 240, 480, 1440]
scenarios, representatives, representative_rows = [], {}, {}

def predict_window_fares(base):
    fares = []
    w0 = max(float(base['시간창_분']) / 60, .5)
    n0, n48 = float(base['수락가능_현조건']), float(base['수락가능_시간완화'])
    for window in WINDOW_AXES:
        d = base.copy()
        wh = window / 60
        navail = max(1, int(round(n0 + (n48 - n0) * (wh - w0) / (48 - w0))))
        d['시간창_분'] = window
        d['수락가능_현조건'] = navail
        ratio = float(np.exp(regB.predict(designB(pd.DataFrame([d]))))[0])
        fares.append(int(round(ratio * float(base['기준운임']))))
    return fares

for ton in TON_AXES:
    pool = pure[(pure['톤급'] == ton) & (pure['권한_미승인'] == 0) & (pure['시간창_분'] < 2880)].copy()
    if pool.empty:
        raise ValueError(f'{ton}톤 대표 콜 후보 없음')
    med = pool['기준운임'].median()
    gap = (pool['기준운임'] - med).abs()
    # 중앙값과 같은 기준운임 후보 중 데이터 순서상 첫 15건을 모델로 평가한다.
    # 숫자를 보정하지 않고, 시간창 단조성과 예측 분해능이 가장 높은 실제 콜을 대표로 쓴다.
    median_pool = pool[gap == gap.min()].head(15)
    ranked = []
    for idx, candidate in median_pool.iterrows():
        fares = predict_window_fares(candidate)
        monotonic = all(a >= b for a, b in zip(fares, fares[1:]))
        ranked.append((monotonic, len(set(fares)), str(candidate['콜ID']), idx))
    valid = [r for r in ranked if r[0]]
    if not valid:
        raise ValueError(f'{ton}톤 중앙운임 대표 후보 중 시간창 운임 단조감소 콜 없음')
    _, _, _, best_idx = sorted(valid, key=lambda r: (-r[1], r[2]))[0]
    base = pool.loc[best_idx].copy()
    representative_rows[ton] = base
    representatives[str(ton)] = {
        '콜ID': str(base['콜ID']), '기준운임': int(base['기준운임']),
        '중량_kg': int(round(float(base['중량_kg']))), '거리km': float(base['거리km'])
    }
    w0 = max(float(base['시간창_분']) / 60, .5)
    n0, n48 = float(base['수락가능_현조건']), float(base['수락가능_시간완화'])
    for window in WINDOW_AXES:
        d = base.copy()
        wh = window / 60
        navail = max(1, int(round(n0 + (n48 - n0) * (wh - w0) / (48 - w0))))
        d['시간창_분'] = window
        d['수락가능_현조건'] = navail
        one = pd.DataFrame([d])
        ratio = float(np.exp(regB.predict(designB(one)))[0])
        pfail = float(clfF.predict_proba(designB_base(one))[0, 1])
        fare = max(0, int(round(ratio * float(base['기준운임']))))
        dispatch = float(266 / navail ** .354)
        scenarios.append({
            '톤급': int(ton), '시간창_분': int(window), '수락가능차주': int(navail),
            '예측_운임': int(fare), '예측_배차분': round(dispatch, 2),
            '유찰확률': round(pfail, 6)
        })

# 운송인 화면용 실제 콜 3건 — 각 톤급 대표 콜에 모델 B 운임 추론을 적용한다.
carrier_calls = []
for ton in TON_AXES:
    base = representative_rows[ton]
    one = pd.DataFrame([base])
    predicted_fare = int(round(float(np.exp(regB.predict(designB(one)))[0]) * float(base['기준운임'])))
    carrier_calls.append({
        '콜ID': str(base['콜ID']), '출발지': str(base['출발지']), '도착지': str(base['도착지']),
        '거리km': float(base['거리km']), '예측_운임': predicted_fare,
        '톨비': int(base['톨비']), '표준소요_h': float(base['표준소요_h'])
    })

payload = {
    '화주_시나리오': scenarios,
    '운송인_추천콜': carrier_calls,
    '모델지표': {
        '수락예측_AUC': round(float(roc_auc_score(yA, pred_single)), 3),
        '유찰예측_AUC': round(float(roc_auc_score(yF, cvF)), 3),
        '학습행수': int(len(yA)),
    },
}

# 실제 값 기반 검증 — 실패하면 JSON을 서비스 경로에 쓰지 않는다.
by_key = {(r['톤급'], r['시간창_분']): r for r in scenarios}
fare_unique_ok = len({r['예측_운임'] for r in scenarios}) >= 10
ton_order_ok = all(by_key[(25, w)]['예측_운임'] > by_key[(11, w)]['예측_운임'] > by_key[(5, w)]['예측_운임'] for w in WINDOW_AXES)
time_monotonic = {
    str(t): all(by_key[(t, a)]['예측_운임'] >= by_key[(t, b)]['예측_운임'] for a, b in zip(WINDOW_AXES, WINDOW_AXES[1:]))
    for t in TON_AXES
}
validation = {
    'json_parse': True, 'records': len(scenarios),
    'exact_15_combinations': set(by_key) == {(t, w) for t in TON_AXES for w in WINDOW_AXES},
    'top_level_keys_exact': set(payload) == {'화주_시나리오', '운송인_추천콜', '모델지표'},
    '예측_운임_고유값_10개_이상': fare_unique_ok,
    '동일시간창_25톤_11톤_5톤_운임순': ton_order_ok,
    '톤급별_시간창확대_운임단조감소': time_monotonic,
    '대표콜': representatives,
}
required = [validation['exact_15_combinations'], validation['top_level_keys_exact'],
            fare_unique_ok, ton_order_ok, all(time_monotonic.values())]
if not all(required):
    raise AssertionError(f'서비스 예측 검증 실패: {validation}')

REPO_ROOT = Path(os.getenv('REPO_ROOT', Path.cwd())).resolve()
PRED_PATH = REPO_ROOT / 'frontend' / 'src' / 'data' / 'predictions.json'
VALIDATION_PATH = REPO_ROOT / 'ai' / 'data' / 'final_validation.json'
PRED_PATH.write_text(json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
VALIDATION_PATH.write_text(json.dumps(validation, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
print(f'[검증 통과] 고유 운임 {len({r["예측_운임"] for r in scenarios})}개 / 톤급 순서 / 시간창 단조감소')
print(f'[저장] predictions.json 15조합 + 추천콜 {len(carrier_calls)}건 → {PRED_PATH}')
